# Task11 - [Advanced] Driver Consistency Score

This consistency score combines two factors — the variability of a driver's finishing positions and their reliability in finishing races — weighted 60% and 40% respectively.

**Standard deviation score** (lower variance → higher score):

$$
\text{std\_score} = \frac{1}{1 + \sigma_{\text{position}}}
$$

**DNF (reliability) score:**

$$
\text{dnf\_score} = 1 - \frac{\text{DNF count}}{\text{Total races}}
$$

**Final consistency score:**

$$
\text{Consistency Score} = 0.6 \times \text{std\_score} + 0.4 \times \text{dnf\_score}
$$

Because raw position variance alone made backmarkers who consistently finished near the back look more "consistent" than actual front-runners, the analysis was restricted to each season's **top 10 point scorers**, so the score measures stability among drivers who were already competitive rather than just steady mediocrity. Validating the results against real F1 history (Schumacher's 2002 season, Vettel's 2011 and 2013 seasons, Verstappen's 2023 season) showed the top scores aligned closely with widely recognized dominant seasons, confirming the metric behaves as intended.

In [1]:
# Design your own formula/metric to measure a driver's "consistency" across a season (you decide what factors matter - e.g. variance in finishing position, DNF rate, points per race, etc)
import pandas as pd
import numpy as np

df = pd.read_csv('../data/merged_f1.csv')

subset = df[['year', 'full_name', 'raceId', 'position', 'statusId', 'points']]

df_status = pd.read_csv('../data/status.csv')
subset = subset.merge(df_status, on='statusId', how='left')

finished_pattern = subset['status'].str.contains(r'^\+\d+ Lap', regex=True)
finished_exact = subset['status'] == 'Finished'
subset['is_finished'] = finished_pattern | finished_exact

result = subset.groupby(['year', 'full_name']).agg(
    avg_position=('position', 'mean'),
    std_position=('position', 'std'),
    race_count=('raceId', 'count'),
    dnf_count=('is_finished', lambda x: (~x).sum()),
    total_points=('points', 'sum')
).reset_index()

result['dnf_rate'] = result['dnf_count'] / result['race_count']
result = result[result['race_count'] >= 5]
result = result[result['total_points'] > 0]

result = result.sort_values(['year', 'total_points'], ascending=[True, False])
result = result.groupby('year').head(10)

result['std_score'] = 1 / (1 + result['std_position'])
result['dnf_score'] = 1 - result['dnf_rate']
result['consistency_score'] = (
    result['std_score'] * 0.6 +
    result['dnf_score'] * 0.4
)

result.to_csv('../data/consistency_scores.csv', index=False)

for year in range(2000, 2025):
    year_data = result[result['year'] == year]
    top5 = year_data.sort_values('consistency_score', ascending=False).head(5)
    print(f"=== {year}년 Top 5 ===")
    print(top5[['full_name', 'consistency_score', 'total_points']])
    print()

=== 2000년 Top 5 ===
                 full_name  consistency_score  total_points
2635    Rubens Barrichello           0.607830          62.0
2627    Michael Schumacher           0.579489         108.0
2628         Mika Häkkinen           0.575582          89.0
2614       David Coulthard           0.555189          73.0
2617  Giancarlo Fisichella           0.408036          18.0

=== 2001년 Top 5 ===
               full_name  consistency_score  total_points
2652  Michael Schumacher           0.680129         123.0
2637     David Coulthard           0.568922          65.0
2659  Rubens Barrichello           0.442723          56.0
2654       Nick Heidfeld           0.441068          12.0
2644  Jacques Villeneuve           0.431105          12.0

=== 2002년 Top 5 ===
               full_name  consistency_score  total_points
2677  Michael Schumacher           0.770749         144.0
2683  Rubens Barrichello           0.502675          77.0
2674  Juan Pablo Montoya           0.483090          50.

# Task12 - [Advanced] Constructor Championship Simulation

In [1]:
# Using historical points-per-position rules, write a function that recalculates the final constructor standings for a given year if the points system were different (e.g. apply 2020 rules to 1990 season, or design your own points scale)
# Compare original standings vs simulated standings for at least 1 season
import pandas as pd
import numpy as np

df = pd.read_csv('../data/merged_f1.csv')

f1_points_systems = {
    # 1950-1957: top 5 finishers, fastest lap bonus point existed separately
    (1950, 1957): {1: 8, 2: 6, 3: 4, 4: 3, 5: 2},

    # 1958-1959: top 5 finishers, shared-points rule removed
    (1958, 1959): {1: 8, 2: 6, 3: 4, 4: 3, 5: 2},

    # 1960: top 6 finishers, fastest lap bonus removed
    (1960, 1960): {1: 8, 2: 6, 3: 4, 4: 3, 5: 2, 6: 1},

    # 1961-1990: top 6 finishers, win worth 9 points (longest-running system)
    (1961, 1990): {1: 9, 2: 6, 3: 4, 4: 3, 5: 2, 6: 1},

    # 1991-2002: top 6 finishers, win worth 10 points
    (1991, 2002): {1: 10, 2: 6, 3: 4, 4: 3, 5: 2, 6: 1},

    # 2003-2009: expanded to top 8 finishers
    (2003, 2009): {1: 10, 2: 8, 3: 6, 4: 5, 5: 4, 6: 3, 7: 2, 8: 1},

    # 2010-2024: top 10 finishers, win worth 25 points (current era)
    (2010, 2024): {1: 25, 2: 18, 3: 15, 4: 12, 5: 10, 6: 8, 7: 6, 8: 4, 9: 2, 10: 1},
}

def get_points_system(year):
    # Look up which points system applies to a given year
    for (start, end), system in f1_points_systems.items():
        if start <= year <= end:
            return system
    raise ValueError(f"No points system found for year {year}")

def calculate_points(position, points_system):
    if pd.isna(position):
        return 0
    position = int(position)
    return points_system.get(position, 0)

def simulate_standings(year, points_system):
    # Filter data for the given year and copy to avoid warnings
    season_data = df[df['year'] == year].copy()
    
    # Apply the points system to each row's finishing position
    season_data['simulated_points'] = season_data['position'].apply(
        lambda x: calculate_points(x, points_system)
    )
    
    # Sum up the ORIGINAL points per constructor (real historical result)
    original_standings = season_data.groupby('constructorId')['points'].sum().reset_index()
    original_standings = original_standings.rename(columns={'points': 'original_points'})
    
    # Sum up the SIMULATED points per constructor (using the new system)
    simulated_standings = season_data.groupby('constructorId')['simulated_points'].sum().reset_index()
    
    # Merge both standings side by side on constructor name
    comparison = original_standings.merge(simulated_standings, on='constructorId', how='left')
    
    # Rank both columns (higher points = better rank, i.e. rank 1)
    comparison['original_rank'] = comparison['original_points'].rank(ascending=False, method='min').astype(int)
    comparison['simulated_rank'] = comparison['simulated_points'].rank(ascending=False, method='min').astype(int)
    
    # Sort by original rank for readability
    comparison = comparison.sort_values('original_rank')
    
    return comparison

# Ask the user which year they want to recalculate
year = int(input("Enter the year to recalculate: "))

# Ask which year's points system to apply
system_year = int(input("Which year's points system do you want to apply? (e.g. 2020): "))
points_system = get_points_system(system_year)   

result = simulate_standings(year, points_system)

# Output both tables side by side
constructors_lookup = df[['constructorId', 'name']].drop_duplicates()
result_named = result.merge(constructors_lookup, on='constructorId', how='left')
final_table = result_named[['name', 'original_points', 'original_rank', 'simulated_points', 'simulated_rank']]
final_table['rank_change'] = final_table['original_rank'] - final_table['simulated_rank']
final_table

,name,original_points,original_rank,simulated_points,simulated_rank,rank_change
0,McLaren,121.0,1,388,1,0
1,Ferrari,110.0,2,343,2,0
2,Benetton,71.0,3,278,3,0
3,Williams,57.0,4,235,4,0
4,Tyrrell,16.0,5,100,5,0
5,Larrousse,11.0,6,77,6,0
6,Leyton House,7.0,7,47,7,0
7,Team Lotus,3.0,8,43,8,0
8,Brabham,2.0,9,18,11,-2
9,Arrows,2.0,9,34,9,0


# Task13 - Circuit Difficulty Analysis

# Circuit Difficulty / Unpredictability Analysis — Methodology Summary

## 1. Objective

The goal was to rank F1 circuits by how "difficult" or "unpredictable" they are,
using only the data available in the dataset: finishing status (DNF reasons),
grid vs. finish position, lap times, and pit stops.

Early on, we distinguished two related but different concepts:
- **Difficulty**: how physically challenging a circuit is to drive
- **Unpredictability**: how much race results deviate from expectations

Given the available data (results, status, lap times, pit stops), the metric we
built turned out to measure **unpredictability** more directly than physical
difficulty — this becomes clear later in the findings.

---

## 2. Key Variables — What We Chose and Why

We started with 6 candidate variables, computed per circuit.

| Variable | What it measures | Why we considered it |
|---|---|---|
| `accident_rate` | Share of results that ended in a crash-related status (Accident, Collision, Spun off, Fatal accident, Collision damage, Debris, Damage) | Most direct signal of circuit-caused danger |
| `mechanical_rate` | Share of results that ended in a mechanical failure (Engine, Gearbox, Brakes, etc.) | Could reflect how harsh a circuit is on cars (bumpy surface, heat, etc.) |
| `avg_laps_behind` | Average number of laps a *finisher* was behind the leader (`+N Laps` statuses) | Captures cases where a driver technically "finished" but was far off competitive pace — a signal `is_finished` alone misses |
| `avg_position_change` | Average absolute difference between grid position and finish position | Directly measures how much race results deviate from qualifying expectations |
| `laptime_std` | Standard deviation of lap times per circuit | Higher variability could indicate more incidents, safety car periods, or pace disruption |
| `avg_pitstops` | Average pit stops per race per circuit | Could reflect tire wear or unexpected race incidents (debris, punctures) |

### How each was calculated

```python
# Status categorization (grouping raw status strings into buckets)
def categorize_status(status):
    if status == 'Finished':
        return 'finished_clean'
    elif status.startswith('+') and 'Lap' in status:
        return 'finished_behind'
    elif status in ['Accident', 'Collision', 'Spun off', 'Fatal accident',
                    'Collision damage', 'Debris', 'Damage']:
        return 'accident'
    elif status in ['Disqualified', 'Withdrew', 'Not classified', '107% Rule',
                    'Did not qualify', 'Did not prequalify', 'Excluded',
                    'Injury', 'Injured', 'Illness', 'Driver unwell',
                    'Safety concerns', 'Not restarted', 'Underweight',
                    'Safety belt', 'Safety', 'Eye injury']:
        return 'administrative'
    else:
        return 'mechanical'

# accident_rate / mechanical_rate = category count / total results, per circuit
# avg_laps_behind = mean of laps extracted from '+N Lap(s)' status strings
# avg_position_change = mean(abs(grid - position))
# laptime_std = std of lap_times.milliseconds, joined to circuit via raceId
# avg_pitstops = mean count of pit_stops rows per race, joined to circuit via raceId
```

### Why we dropped `laptime_std` and `avg_pitstops`

A first PCA run (all 6 features, all-time data) showed these two contributed
almost nothing:

| Feature | Loading (PC1) |
|---|---|
| accident_rate | 0.140 |
| mechanical_rate | 0.601 |
| avg_laps_behind | 0.503 |
| avg_position_change | 0.580 |
| laptime_std | 0.174 |
| avg_pitstops | -0.020 |

Explained variance: **41.6%**

We re-ran PCA with only the 4 strongest features (`accident_rate`,
`mechanical_rate`, `avg_laps_behind`, `avg_position_change`):

- Explained variance improved to **54.2%**
- Correlation between the 6-feature score and the 4-feature score: **0.99**

This confirmed `laptime_std` and `avg_pitstops` were adding noise rather than
signal, and were dropped from the final model. (Separately, both variables
also had heavy missing data in older eras — `lap_times.csv` and
`pit_stops.csv` only contain modern-era records — which made them
incompatible with a phase-based analysis anyway.)

---

## 3. Why PCA — and How It Was Used

### The problem
We had 4 variables per circuit and wanted to combine them into a single
"score." A manual weighted sum (like we used for the driver consistency
score) requires deciding weights by hand, which is subjective.

### The solution: PCA (Principal Component Analysis)
PCA finds the direction in the data along which circuits differ from each
other the most, and expresses each circuit as a single value along that
direction (the first principal component, PC1). The "weight" each original
variable gets in that combined score (the **loading**) is derived
mathematically from the data's variance structure, not chosen by hand.

### Steps taken
```python
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

feature_cols = ['accident_rate', 'mechanical_rate', 'avg_laps_behind', 'avg_position_change']

# 1. Standardize (mean 0, std 1) so no feature dominates due to scale
scaled = StandardScaler().fit_transform(circuit_stats[feature_cols])

# 2. Reduce to a single score
pca = PCA(n_components=1)
circuit_stats['difficulty_score'] = pca.fit_transform(scaled)[:, 0]

# 3. Inspect which variables drove the score
loadings = pd.DataFrame(pca.components_.T, columns=['PC1'], index=feature_cols)
```

### All-time (no phase split) result

Explained variance: 54.2%. Top 10 circuits by score:

| Circuit | Score |
|---|---|
| Zeltweg | 4.74 |
| Circuit de Pedralbes | 2.34 |
| Reims-Gueux | 2.24 |
| Indianapolis Motor Speedway | 2.13 |
| Phoenix street circuit | 1.98 |
| Donington Park | 1.87 |
| Detroit Street Circuit | 1.80 |
| Aintree | 1.76 |
| Montjuïc | 1.59 |
| Brands Hatch | 1.46 |

**Problem identified**: this list is dominated almost entirely by circuits
from F1's earliest era (1950s–60s), and notably excludes circuits widely
regarded as difficult today (e.g. Monaco). This pointed to a confound: the
score was capturing *how dangerous/unpredictable an era was*, not *how
difficult a specific circuit is*, since early-era cars were universally less
reliable and circuits had almost no safety infrastructure.

---

## 4. Why We Needed to Split by Era ("Safety Phase")

Comparing a 1960s race directly to a 2020s race is not a fair comparison —
car reliability and circuit safety standards were fundamentally different.
Without splitting by era, "circuit difficulty" and "how dangerous racing was
in general at that time" get mixed together and can't be separated.

### Phase definition

```python
def get_safety_phase(year):
    # before 1978: almost no circuit safety standards
    # 1978-1993: gradual safety improvements (post Ronnie Peterson fatal crash),
    #            but still the high-risk ground-effect/turbo era
    # 1994+: FIA overhaul of circuit and car safety after the Senna/Ratzenberger
    #        fatal accidents at Imola
    if year < 1978:
        return 'badSafety'
    elif year < 1994:
        return 'normalSafety'
    else:
        return 'goodSafety'
```

### Minimum sample size filter
To avoid a circuit's score being distorted by just 1–2 races, we required at
least 5 races within a given phase for a (phase, circuit) pair to be
included:

```python
MIN_RACES = 5
valid_circuits = phase_stats[phase_stats['race_count'] >= MIN_RACES]
```

### Re-running PCA within each phase
The same 4-feature PCA was re-run **separately for each phase**, so circuits
are only compared against others from the same era:

```python
for phase in circuit_stats_phase['safety_phase'].unique():
    phase_data = circuit_stats_phase[circuit_stats_phase['safety_phase'] == phase].copy()
    scaled = StandardScaler().fit_transform(phase_data[feature_cols])
    pca_phase = PCA(n_components=1)
    phase_data['difficulty_score'] = pca_phase.fit_transform(scaled)[:, 0]
```

---

## 5. Results After Splitting by Phase

### badSafety (pre-1978) — Top 10
| Circuit | Score |
|---|---|
| Indianapolis Motor Speedway | 2.27 |
| Mosport International Raceway | 1.74 |
| Autódromo José Carlos Pace | 1.56 |
| Circuit de Monaco | 1.56 |
| Red Bull Ring | 1.31 |
| Jarama | 1.18 |
| Nürburgring | 0.77 |
| Scandinavian Raceway | 0.41 |
| Circuit Park Zandvoort | 0.34 |
| Kyalami | 0.10 |

### goodSafety (1994+) — Top 10
| Circuit | Score |
|---|---|
| Indianapolis Motor Speedway | 3.28 |
| Circuit Gilles Villeneuve | 2.00 |
| Hockenheimring | 1.79 |
| Albert Park Grand Prix Circuit | 1.76 |
| Circuit de Monaco | 1.75 |
| Autodromo Enzo e Dino Ferrari | 1.55 |
| Autódromo José Carlos Pace | 1.34 |
| Nürburgring | 1.30 |
| Sepang International Circuit | 1.22 |
| Circuit de Nevers Magny-Cours | 0.92 |

### normalSafety (1978–1993) — Top 10
| Circuit | Score |
|---|---|
| Long Beach | 3.76 |
| Zolder | 1.51 |
| Circuit de Monaco | 1.01 |
| Detroit Street Circuit | 0.97 |
| Hungaroring | 0.86 |
| Autódromo Internacional Nelson Piquet | 0.80 |
| Suzuka Circuit | 0.77 |
| Adelaide Street Circuit | 0.60 |
| Kyalami | 0.54 |
| Autódromo Hermanos Rodríguez | 0.38 |

### What improved
- Circuits are now compared fairly within their own era, removing the
  "1950s circuits dominate everything" distortion.
- Monaco consistently appears across all three phases (rank 3–5), which
  aligns with its real-world reputation as a persistently tricky circuit.
- Indianapolis Motor Speedway ranks #1 in two separate eras — a genuinely
  interesting, era-independent finding (likely due to being an oval-track
  venue retrofitted for F1, unlike purpose-built road circuits).

### Remaining open questions
- Long Beach's outlier score (3.76, far above 2nd place) needs verification
  against its actual race count in that phase — could be a small-sample
  distortion rather than a real signal.
- Some historically notorious circuits (e.g. the old Nürburgring
  Nordschleife, old Spa-Francorchamps) don't appear prominently — this may
  be because they didn't meet the `MIN_RACES = 5` threshold in that phase,
  not because the metric judged them "safe."
- `mechanical_rate` still appears to be a major driver of the score in most
  phases, which raises the question of whether we're measuring circuit
  danger or car reliability at that venue.

In [20]:
import re
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# ============================================================
# 1. Load raw data files
# ============================================================
df_results = pd.read_csv('../data/results.csv')
df_races = pd.read_csv('../data/races.csv')
df_circuits = pd.read_csv('../data/circuits.csv')
df_status = pd.read_csv('../data/status.csv')
df_lap_times = pd.read_csv('../data/lap_times.csv')
df_pit_stops = pd.read_csv('../data/pit_stops.csv')

# ============================================================
# 2. Merge results + races + circuits + status into one table
# ============================================================
results_subset = df_results[['raceId', 'grid', 'position', 'statusId']]
races_subset = df_races[['raceId', 'circuitId', 'year']]
circuits_subset = df_circuits[['circuitId', 'name']].rename(columns={'name': 'circuitName'})

df = results_subset.merge(races_subset, on='raceId', how='left')
df = df.merge(circuits_subset, on='circuitId', how='left')
df = df.merge(df_status, on='statusId', how='left')


# ============================================================
# 3. Categorize status into: finished_clean, finished_behind,
#    accident, administrative, mechanical
# ============================================================
def categorize_status(status):
    if status == 'Finished':
        return 'finished_clean'
    elif status.startswith('+') and 'Lap' in status:
        return 'finished_behind'
    elif status in ['Accident', 'Collision', 'Spun off', 'Fatal accident',
                    'Collision damage', 'Debris', 'Damage']:
        return 'accident'
    elif status in ['Disqualified', 'Withdrew', 'Not classified', '107% Rule',
                    'Did not qualify', 'Did not prequalify', 'Excluded',
                    'Injury', 'Injured', 'Illness', 'Driver unwell',
                    'Safety concerns', 'Not restarted', 'Underweight',
                    'Safety belt', 'Safety', 'Eye injury']:
        return 'administrative'
    else:
        return 'mechanical'


df['status_category'] = df['status'].apply(categorize_status)


# ============================================================
# 4. Derived columns: laps_behind, position_change
# ============================================================
def extract_laps_behind(status):
    match = re.match(r'^\+(\d+) Lap', status)
    if match:
        return int(match.group(1))
    elif status == 'Finished':
        return 0
    else:
        return np.nan


df['laps_behind'] = df['status'].apply(extract_laps_behind)
df['position'] = pd.to_numeric(df['position'], errors='coerce')
df['position_change'] = abs(df['grid'] - df['position'])

# ============================================================
# 5. Aggregate per circuit (ALL-TIME, no phase split)
# ============================================================
category_counts = df.groupby(['circuitName', 'status_category']).size().unstack(fill_value=0)
category_counts['total'] = category_counts.sum(axis=1)
category_counts['accident_rate'] = category_counts['accident'] / category_counts['total']
category_counts['mechanical_rate'] = category_counts['mechanical'] / category_counts['total']

avg_laps_behind = df.groupby('circuitName')['laps_behind'].mean()
avg_position_change = df.groupby('circuitName')['position_change'].mean()

lap_join = df_lap_times.merge(df[['raceId', 'circuitName']].drop_duplicates(), on='raceId', how='left')
laptime_std = lap_join.groupby('circuitName')['milliseconds'].std()

pit_join = df_pit_stops.merge(df[['raceId', 'circuitName']].drop_duplicates(), on='raceId', how='left')
pitstop_count = pit_join.groupby(['circuitName', 'raceId']).size().groupby('circuitName').mean()

circuit_stats = category_counts[['accident_rate', 'mechanical_rate', 'total']].reset_index()
circuit_stats = circuit_stats.merge(avg_laps_behind.reset_index(name='avg_laps_behind'), on='circuitName')
circuit_stats = circuit_stats.merge(avg_position_change.reset_index(name='avg_position_change'), on='circuitName')
circuit_stats = circuit_stats.merge(laptime_std.reset_index(name='laptime_std'), on='circuitName', how='left')
circuit_stats = circuit_stats.merge(pitstop_count.reset_index(name='avg_pitstops'), on='circuitName', how='left')

# Drop circuits with too few results to trust the statistics
circuit_stats = circuit_stats[circuit_stats['total'] >= 20]

# ============================================================
# 6. v1: PCA with all 6 features
# ============================================================
feature_cols_v1 = ['accident_rate', 'mechanical_rate', 'avg_laps_behind',
                    'avg_position_change', 'laptime_std', 'avg_pitstops']

circuit_stats_clean = circuit_stats.dropna(subset=feature_cols_v1).copy()

scaled = StandardScaler().fit_transform(circuit_stats_clean[feature_cols_v1])

pca = PCA(n_components=1)
circuit_stats_clean['difficulty_score'] = pca.fit_transform(scaled)[:, 0]

loadings = pd.DataFrame(pca.components_.T, columns=['PC1'], index=feature_cols_v1)
print("=== v1: all 6 features ===")
print(loadings)
print("Explained variance:", pca.explained_variance_ratio_)
print()

# ============================================================
# 7. v2: PCA with 4 features (dropped avg_pitstops, laptime_std -
#    both had near-zero loadings in v1)
# ============================================================
feature_cols_v2 = ['accident_rate', 'mechanical_rate', 'avg_laps_behind', 'avg_position_change']

circuit_stats_clean_v2 = circuit_stats.dropna(subset=feature_cols_v2).copy()

scaled_v2 = StandardScaler().fit_transform(circuit_stats_clean_v2[feature_cols_v2])

pca_v2 = PCA(n_components=1)
circuit_stats_clean_v2['difficulty_score_v2'] = pca_v2.fit_transform(scaled_v2)[:, 0]

loadings_v2 = pd.DataFrame(pca_v2.components_.T, columns=['PC1'], index=feature_cols_v2)
print("=== v2: 4 features (dropped avg_pitstops, laptime_std) ===")
print(loadings_v2)
print("Explained variance:", pca_v2.explained_variance_ratio_)
print()

# ============================================================
# 8. Compare v1 vs v2 rankings (are they still similar?)
# ============================================================
comparison = circuit_stats_clean_v2[['circuitName', 'difficulty_score_v2']].merge(
    circuit_stats_clean[['circuitName', 'difficulty_score']], on='circuitName'
)
print("Correlation between v1 and v2 scores:")
print(comparison[['difficulty_score_v2', 'difficulty_score']].corr())
print()

# ============================================================
# 9. Top 10 circuits (all-time, v2 - the version we ended up trusting)
# ============================================================
print("=== All-time Top 10 (v2, no phase split) ===")
print(
    circuit_stats_clean_v2.sort_values('difficulty_score_v2', ascending=False)
    [['circuitName', 'difficulty_score_v2']].head(10)
)

=== v1: all 6 features ===
                          PC1
accident_rate        0.139770
mechanical_rate      0.600612
avg_laps_behind      0.503071
avg_position_change  0.579516
laptime_std          0.174338
avg_pitstops        -0.020396
Explained variance: [0.41606021]

=== v2: 4 features (dropped avg_pitstops, laptime_std) ===
                          PC1
accident_rate        0.129945
mechanical_rate      0.604283
avg_laps_behind      0.537228
avg_position_change  0.573884
Explained variance: [0.54187702]

Correlation between v1 and v2 scores:
                     difficulty_score_v2  difficulty_score
difficulty_score_v2             1.000000          0.990509
difficulty_score                0.990509          1.000000

=== All-time Top 10 (v2, no phase split) ===
                    circuitName  difficulty_score_v2
75                      Zeltweg             4.736475
27         Circuit de Pedralbes             2.342538
62                  Reims-Gueux             2.242250
39  Indianapo

In [21]:
import re
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# ============================================================
# 1. Load raw data files
# ============================================================
df_results = pd.read_csv('../data/results.csv')
df_races = pd.read_csv('../data/races.csv')
df_circuits = pd.read_csv('../data/circuits.csv')
df_status = pd.read_csv('../data/status.csv')
df_lap_times = pd.read_csv('../data/lap_times.csv')
df_pit_stops = pd.read_csv('../data/pit_stops.csv')

# ============================================================
# 2. Merge results + races + circuits + status into one table
# ============================================================
results_subset = df_results[['raceId', 'grid', 'position', 'statusId']]
races_subset = df_races[['raceId', 'circuitId', 'year']]
circuits_subset = df_circuits[['circuitId', 'name']].rename(columns={'name': 'circuitName'})

df = results_subset.merge(races_subset, on='raceId', how='left')
df = df.merge(circuits_subset, on='circuitId', how='left')
df = df.merge(df_status, on='statusId', how='left')


# ============================================================
# 3. Categorize status into: finished_clean, finished_behind,
#    accident, administrative, mechanical
# ============================================================
def categorize_status(status):
    if status == 'Finished':
        return 'finished_clean'
    elif status.startswith('+') and 'Lap' in status:
        return 'finished_behind'
    elif status in ['Accident', 'Collision', 'Spun off', 'Fatal accident',
                    'Collision damage', 'Debris', 'Damage']:
        return 'accident'
    elif status in ['Disqualified', 'Withdrew', 'Not classified', '107% Rule',
                    'Did not qualify', 'Did not prequalify', 'Excluded',
                    'Injury', 'Injured', 'Illness', 'Driver unwell',
                    'Safety concerns', 'Not restarted', 'Underweight',
                    'Safety belt', 'Safety', 'Eye injury']:
        return 'administrative'
    else:
        return 'mechanical'


df['status_category'] = df['status'].apply(categorize_status)


# ============================================================
# 4. Derived columns: laps_behind, position_change
# ============================================================
def extract_laps_behind(status):
    match = re.match(r'^\+(\d+) Lap', status)
    if match:
        return int(match.group(1))
    elif status == 'Finished':
        return 0
    else:
        return np.nan


df['laps_behind'] = df['status'].apply(extract_laps_behind)
df['position'] = pd.to_numeric(df['position'], errors='coerce')
df['position_change'] = abs(df['grid'] - df['position'])


# ============================================================
# 5. Assign safety phase based on year
# ============================================================
def get_safety_phase(year):
    # before 1978: almost no circuit safety standards
    # 1978-1993: gradual safety improvements (post Ronnie Peterson crash), still risky turbo/ground-effect era
    # 1994+: FIA overhaul after Senna/Ratzenberger fatal accidents at Imola
    if year < 1978:
        return 'badSafety'
    elif year < 1994:
        return 'normalSafety'
    else:
        return 'goodSafety'


df['safety_phase'] = df['year'].apply(get_safety_phase)

# ============================================================
# 6. Filter to circuits with enough races within each phase
# ============================================================
phase_stats = df.groupby(['safety_phase', 'circuitName'])['raceId'].nunique().reset_index(name='race_count')

MIN_RACES = 5
valid_circuits = phase_stats[phase_stats['race_count'] >= MIN_RACES]

# ============================================================
# 7. Compute accident_rate, mechanical_rate, avg_laps_behind, avg_position_change
#    grouped by (safety_phase, circuitName)
# ============================================================
category_counts_phase = df.groupby(['safety_phase', 'circuitName', 'status_category']).size().unstack(fill_value=0)
category_counts_phase['total'] = category_counts_phase.sum(axis=1)
category_counts_phase['accident_rate'] = category_counts_phase['accident'] / category_counts_phase['total']
category_counts_phase['mechanical_rate'] = category_counts_phase['mechanical'] / category_counts_phase['total']

avg_laps_behind_phase = df.groupby(['safety_phase', 'circuitName'])['laps_behind'].mean()
avg_position_change_phase = df.groupby(['safety_phase', 'circuitName'])['position_change'].mean()

circuit_stats_phase = category_counts_phase[['accident_rate', 'mechanical_rate']].reset_index()
circuit_stats_phase = circuit_stats_phase.merge(
    avg_laps_behind_phase.reset_index(name='avg_laps_behind'), on=['safety_phase', 'circuitName']
)
circuit_stats_phase = circuit_stats_phase.merge(
    avg_position_change_phase.reset_index(name='avg_position_change'), on=['safety_phase', 'circuitName']
)

# ============================================================
# 8. Compute laptime_std and avg_pitstops grouped by (safety_phase, circuitName)
# ============================================================
race_phase_lookup = df[['raceId', 'circuitName', 'safety_phase']].drop_duplicates()

lap_join_phase = df_lap_times.merge(race_phase_lookup, on='raceId', how='left')
laptime_std_phase = lap_join_phase.groupby(['safety_phase', 'circuitName'])['milliseconds'].std()

pit_join_phase = df_pit_stops.merge(race_phase_lookup, on='raceId', how='left')
pitstop_count_phase = (
    pit_join_phase.groupby(['safety_phase', 'circuitName', 'raceId']).size()
    .groupby(['safety_phase', 'circuitName']).mean()
)

circuit_stats_phase_full = circuit_stats_phase.merge(
    laptime_std_phase.reset_index(name='laptime_std'), on=['safety_phase', 'circuitName'], how='left'
)
circuit_stats_phase_full = circuit_stats_phase_full.merge(
    pitstop_count_phase.reset_index(name='avg_pitstops'), on=['safety_phase', 'circuitName'], how='left'
)

# ============================================================
# 9. Keep only (phase, circuit) combos with enough races
# ============================================================
circuit_stats_phase_full = circuit_stats_phase_full.merge(
    valid_circuits[['safety_phase', 'circuitName']],
    on=['safety_phase', 'circuitName'],
    how='inner'
)

print("Missing values per column:")
print(circuit_stats_phase_full.isnull().sum())
print()

# ============================================================
# 10. Run PCA separately within each safety phase, using all 6 features
# ============================================================
feature_cols_all6 = ['accident_rate', 'mechanical_rate', 'avg_laps_behind',
                    'avg_position_change']

results_by_phase = []

for phase in circuit_stats_phase_full['safety_phase'].unique():
    phase_data = circuit_stats_phase_full[circuit_stats_phase_full['safety_phase'] == phase].copy()
    phase_data_clean = phase_data.dropna(subset=feature_cols_all6)

    if len(phase_data_clean) < 5:
        print(f"{phase}: skipped, not enough complete rows ({len(phase_data_clean)})")
        continue

    scaled = StandardScaler().fit_transform(phase_data_clean[feature_cols_all6])

    pca_phase = PCA(n_components=1)
    phase_data_clean['difficulty_score'] = pca_phase.fit_transform(scaled)[:, 0]

    loadings = pd.DataFrame(pca_phase.components_.T, columns=['PC1'], index=feature_cols_all6)
    print(f"=== {phase} (n={len(phase_data_clean)}) ===")
    print(loadings)
    print("Explained variance:", pca_phase.explained_variance_ratio_)
    print()

    results_by_phase.append(phase_data_clean)

circuit_stats_final = pd.concat(results_by_phase, ignore_index=True)

# ============================================================
# 11. Show top 5 most "difficult" circuits per phase
# ============================================================
for phase in circuit_stats_final['safety_phase'].unique():
    print(f"=== {phase} Top 10 ===")
    top5 = circuit_stats_final[circuit_stats_final['safety_phase'] == phase] \
        .sort_values('difficulty_score', ascending=False).head(10)
    print(top5[['circuitName', 'difficulty_score']])
    print()

Missing values per column:
safety_phase            0
circuitName             0
accident_rate           0
mechanical_rate         0
avg_laps_behind         0
avg_position_change     0
laptime_std            44
avg_pitstops           46
dtype: int64

=== badSafety (n=21) ===
                          PC1
accident_rate        0.687594
mechanical_rate     -0.569922
avg_laps_behind     -0.287631
avg_position_change  0.345936
Explained variance: [0.45283117]

=== goodSafety (n=27) ===
                          PC1
accident_rate        0.513661
mechanical_rate      0.549791
avg_laps_behind      0.376413
avg_position_change  0.540551
Explained variance: [0.61888231]

=== normalSafety (n=23) ===
                          PC1
accident_rate        0.657604
mechanical_rate     -0.542331
avg_laps_behind      0.274632
avg_position_change  0.444984
Explained variance: [0.42032951]

=== badSafety Top 10 ===
                      circuitName  difficulty_score
10    Indianapolis Motor Speedway          